# demo_rl18

> Phase 18, Part B (`specs/multiagent/phase18/`): clones of one 3B model trained with team reward on the colony board against the plain board, on the Mac from the Colab outputs: H18-B1 (team reward trains better on the colony board), H18-B2 (an emergent division of labour), H18-B3 (robustness learned at 4-8 agents holds at 32), H18-B4 (goal drift), under `RL18_PLAN`. The speed check is read on Colab.

Artifacts (gitignored): `results/multiagent/rl/{quick_p18, eval_p18}/`. Input: `explRL_rl18.zip` unzipped into `results/multiagent/rl/colab/` (`runs_p18/`, `eval_p18/`).

- **Quick** (normal tests): synthetic evaluation rows end to end.
- **Main** (`--flags train`): only when the Colab output is present and `RL18_PLAN` is frozen.

In [ ]:
import json
import subprocess

from explRL.bench_grid import readCSV
from explRL.coop_grid import coopHash
from explRL.rl18_grid import RL18_PLAN
from explRL.rl18_report import figRL18, rl18Summaries, syntheticRows18B, writeRL18
from explRL.train import demo_artifact_dir

RL = demo_artifact_dir(None, 'multiagent/rl')
COLAB = RL / 'colab'


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def analyse(mode, rows, out, nboot=2000):
    out.mkdir(parents=True, exist_ok=True)
    S = rl18Summaries(rows, nboot=nboot)
    meta = dict(mode=mode, planHash=coopHash(RL18_PLAN), commit=gitCommit())
    writeRL18(out / 'report.md', S, meta)
    figRL18(out, S)
    (out / 'summary.json').write_text(json.dumps(dict(meta, **S), indent=1, sort_keys=True, default=str) + '\n')
    return S

## Quick: synthetic rows

In [ ]:
qo = RL / 'quick_p18'
S = analyse('quick', syntheticRows18B(), qo, nboot=200)
assert all((qo / f).stat().st_size > 0 for f in ('report.md', 'rl18.png', 'summary.json'))
assert all(S['hyp'][k]['supported'] for k in ('H18_B1', 'H18_B2', 'H18_B3'))
print(open(qo / 'report.md').read()[:1500])

## Main (`--flags train`): `RL18_PLAN`'s analysis

In [ ]:
#| train
p = COLAB / 'eval_p18' / 'runs.csv'
if not p.exists():
    print('skipped: unzip explRL_rl18.zip into results/multiagent/rl/colab/ first')
else:
    assert RL18_PLAN['frozen'], 'the analysis uses the frozen RL18_PLAN'
    cut = [json.loads(x)['cut'] for d in sorted((COLAB / 'runs_p18').glob('*_s*')) for x in open(d / 'metrics.jsonl')]
    print('cut prompts in the main runs:', sum(cut), '(must be 0)')
    S = analyse('full', readCSV(p), RL / 'eval_p18')
    print(open(RL / 'eval_p18' / 'report.md').read())